# Sprint E10 walkthrough: dynamic risk allocation and loss management

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "efb").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
ALLOC = DATA / "allocation"

In [2]:
# the data hash in the results file must be the hash of the artifacts
# the criteria are read from, recomputed now, not copied
from efb import evaluate

stored = json.loads(
    (ROOT / "sprints" / "E10" / "RESULTS.json").read_text()
)
assert evaluate.e10_data_hash(DATA) == stored["data_hash"], "artifact hash drift"
print("data_hash", stored["data_hash"])
print("verdicts:", stored["reference_values"]["verdicts"])

data_hash 34cefce18053476daca2b9b40a9525170e15837d14308f15a1ad9f9fb3a16b69
verdicts: {'F10.1': 'fail', 'F10.1b': 'pass', 'F10.2': 'pass', 'F10.2b': 'pass', 'F10.3': 'fail', 'F10.3b': 'pass'}


## 1. Every criterion, its stored number and its verdict

In [3]:
for name, block in stored["criteria"].items():
    print(name, block["verdict"], json.dumps(block["stored_numbers"])[:160])

F10.1 fail {"simulated_median_drawdown": -0.14030834893682936, "analytical_median_drawdown": 0.0354329710482708, "relative_gap_at_median": 2.9598245584793177, "gaussian_me
F10.1b pass {"horizon_years": 14.5, "expected_mdd_at_horizon": 0.1993770805683501, "expected_mdd_relative_gap": 0.2962664086721387, "n_obs": 174}
F10.2 pass {"control_mean_sharpe_diff": -0.487285170300088, "control_improves_sharpe": false, "seed_ew_real_sharpe_diff": 0.0, "seed_ew_n_obs": 207, "seed_ew_n_dates_avail
F10.2b pass {"reentering_control_mean_sharpe_diff": -0.07161747413052198, "reentering_control_improves_sharpe": false, "reentering_real_sharpe_diff": -0.07716140464418775, 
F10.3 fail {"raw_dispersion": 0.27535511464122947, "targeted_dispersion": 0.2338401979010474, "dispersion_reduction": 0.1507686421378931, "n_years_raw": 15, "n_years_targe
F10.3b pass {"daily_21_reduction": 0.286841908887589, "daily_42_reduction": 0.19526090861895973, "daily_63_reduction": 0.16342907720713906, "daily_126_reduction": 0.

## 2. The design book and the Kelly analysis

The (rho, phi) whose net annualized Sharpe is closest to 1.0, and the Kelly fraction f* = mu / sigma^2 with the growth curve.

In [4]:
from efb import allocate

config = allocate.pick_design_config(DATA)
print("design config", config)
kelly = pd.read_parquet(ALLOC / 'kelly.parquet').iloc[0]
full = kelly['mean_ann'] / kelly['vol_ann'] ** 2
assert abs(full - kelly['kelly_full']) < 1e-9
print('full Kelly', round(kelly['kelly_full'], 4))
print('half Kelly', round(kelly['kelly_half'], 4))
print('growth at full Kelly', round(kelly['growth_full'], 4))
print('growth at half Kelly', round(kelly['growth_half'], 4))
print('growth loss overbet', round(kelly['growth_loss_overbet'], 4))

design config {'rho': 0.02, 'phi': 0.95, 'net_sharpe': 1.1738437142793923, 'gross_sharpe': 1.4561053319301573, 'distance': 0.17384371427939227, 'seed': 1, 'net_sharpe_seed': np.float64(0.9781104435409355)}
full Kelly 9.7811
half Kelly 4.8906
growth at full Kelly 0.4784
growth at half Kelly 0.3588
growth loss overbet 0.0359


## 3. F10.1: the drawdown distribution against the analytical median

In [5]:
drawdown = pd.read_parquet(ALLOC / 'drawdown.parquet').iloc[0]
# the analytical median is ln(2) sigma^2 / (2 mu), recomputed by hand
analytical = np.log(2.0) * kelly['vol_ann'] ** 2 / (2.0 * kelly['mean_ann'])
assert abs(analytical - drawdown['analytical_median_drawdown']) < 1e-9
# like with like: the simulated median is signed, the analytical is a
# magnitude, so the gap takes the magnitude of the simulated side
sim_mag = abs(drawdown['simulated_median_drawdown'])
gap = (sim_mag - drawdown['analytical_median_drawdown']) / drawdown['analytical_median_drawdown']
assert abs(gap - drawdown['relative_gap_at_median']) < 1e-9
print('simulated median', round(drawdown['simulated_median_drawdown'], 4))
print('analytical median', round(drawdown['analytical_median_drawdown'], 4))
print('relative gap', round(drawdown['relative_gap_at_median'], 4))
print('gaussian control', round(drawdown['gaussian_median_drawdown'], 4))
assert drawdown['gaussian_median_drawdown'] < drawdown['simulated_median_drawdown'], \
    'the Gaussian control must draw down deeper than the bootstrap'

simulated median -0.1403
analytical median 0.0354
relative gap 2.9598
gaussian control -0.154


## 4. F10.1b: the horizon-matched expected maximum drawdown

In [6]:
# the horizon is n_obs * 21 / 252 years and the expected maximum
# drawdown is the Magdon-Ismail positive-drift value, recomputed
horizon = kelly['n_obs'] * 21 / 252
assert abs(horizon - drawdown['horizon_years']) < 1e-12
x = kelly['mean_ann'] ** 2 * horizon / (2.0 * kelly['vol_ann'] ** 2)
qp = 0.25 * np.log(x) + 0.49088
expected = 2.0 * kelly['vol_ann'] ** 2 / kelly['mean_ann'] * qp
assert abs(expected - drawdown['expected_mdd_at_horizon']) < 1e-9
assert 0.15 <= drawdown['expected_mdd_at_horizon'] <= 0.25
assert drawdown['expected_mdd_relative_gap'] < 1.0
print('horizon years', round(drawdown['horizon_years'], 1))
print('expected max drawdown', round(drawdown['expected_mdd_at_horizon'], 4))
print('expected-vs-simulated gap', round(drawdown['expected_mdd_relative_gap'], 4))

horizon years 14.5
expected max drawdown 0.1994
expected-vs-simulated gap 0.2963


## 5. F10.3: vol targeting and the realized-vol dispersion

In [7]:
voltarget = pd.read_parquet(ALLOC / 'voltarget.parquet').iloc[0]
# both dispersions are computed on the aligned year set
assert voltarget['n_years_raw'] == voltarget['n_years_targeted'] + 1
assert voltarget['n_years_aligned'] == voltarget['n_years_targeted']
print('raw dispersion', round(voltarget['raw_dispersion'], 4))
print('targeted dispersion', round(voltarget['targeted_dispersion'], 4))
print('reduction', round(voltarget['dispersion_reduction'], 4))

raw dispersion 0.2754
targeted dispersion 0.2338
reduction 0.1508


## 6. F10.3b: the daily-return vol estimate sweep

In [8]:
voltarget_daily = pd.read_parquet(ALLOC / 'voltarget_daily.parquet')
print(voltarget_daily.to_string(index=False))
max_reduction = float(voltarget_daily['dispersion_reduction'].max())
assert max_reduction < 0.40, 'no daily window may clear the 40% bar'
assert max_reduction == float(voltarget_daily.iloc[0]['dispersion_reduction'])
print('max daily reduction', round(max_reduction, 4))

estimator  window  raw_dispersion  targeted_dispersion  dispersion_reduction  n_years_aligned  n_years_dropped dropped_years
    daily      21        0.275355             0.196372              0.286842               14                1          2012
    daily      42        0.275355             0.221589              0.195261               14                1          2012
    daily      63        0.275355             0.230354              0.163429               14                1          2012
    daily     126        0.275355             0.219541              0.202698               14                1          2012
    daily     252        0.280606             0.245115              0.126479               13                2     2012,2013
max daily reduction 0.2868


## 7. F10.2: the stop-loss and its i.i.d. control

In [9]:
stoploss = pd.read_parquet(ALLOC / 'stoploss.parquet')
control = stoploss[stoploss['book'] == 'design'].iloc[0]
print(
    'control mean Sharpe diff',
    round(control['control_mean_sharpe_diff'], 4),
)
assert not bool(control['control_improves_sharpe']), 'the i.i.d. control must not improve'
assert not bool(control['reentering_control_improves_sharpe']), \
    'the re-entering stop must not improve on the i.i.d. control'
assert control['n_obs'] == int(kelly['n_obs'])
print(stoploss.to_string(index=False))

control mean Sharpe diff -0.4873
       book  base_sharpe  control_mean_sharpe_diff  control_improves_sharpe  real_book_sharpe_diff  reentering_control_mean_sharpe_diff  reentering_control_improves_sharpe  reentering_real_sharpe_diff  reentering_entries  reentering_exits  reentering_days_flat  n_bootstrap  n_obs  n_dates_available first_date  last_date  stop_fired
     design     0.978110                 -0.487285                    False              -0.394374                            -0.071617                               False                    -0.077161                   2                 3                     9         2000    174                174 2012-02-29 2026-07-31        True
    seed_ew     2.207336                       NaN                    False               0.000000                                  NaN                               False                     0.000000                   0                 0                     0            0    207               4193

## 8. Drawdowns by VIX regime, the risk budget per regime

In [10]:
regime = pd.read_parquet(ALLOC / 'regime.parquet')
print(regime.to_string(index=False))
assert set(regime['vix_tercile']) == {0, 1, 2}

 vix_tercile  mean_vix  max_drawdown  n_underwater  n_obs
           0 12.808793     -0.096651            48     58
           1 16.437241     -0.130857            38     58
           2 24.314828     -0.063602            28     58


## 9. The D9 panel map: which parquet column each panel reads

In [11]:
from dashboard.tabs import d09_risk_allocation as d9

panels = {
    'kelly': d9.kelly_panel(),
    'drawdown': d9.drawdown_panel(),
    'voltarget': d9.voltarget_panel(),
    'stoploss': d9.stoploss_panel(),
    'regime': d9.regime_panel(),
}
for name, panel in panels.items():
    assert not panel.empty, name
    print(name, list(panel.columns))

kelly ['Sharpe (annualized)', 'SE of Sharpe', 'mean (annualized)', 'vol (annualized)', 'full Kelly leverage', 'half Kelly leverage', 'growth at full Kelly', 'growth at half Kelly', 'growth loss when SR overstated by one SE']
drawdown ['simulated_median_drawdown', 'analytical_median_drawdown', 'relative_gap_at_median', 'gaussian_median_drawdown', 'horizon_years', 'expected_mdd_at_horizon', 'expected_mdd_relative_gap', 'n_bootstrap', 'n_obs']
voltarget ['raw_dispersion', 'targeted_dispersion', 'dispersion_reduction', 'raw_mean_vol', 'targeted_mean_vol', 'n_years_raw', 'n_years_targeted', 'n_years_aligned']
stoploss ['book', 'base_sharpe', 'control_mean_sharpe_diff', 'control_improves_sharpe', 'real_book_sharpe_diff', 'reentering_control_mean_sharpe_diff', 'reentering_control_improves_sharpe', 'reentering_real_sharpe_diff', 'reentering_entries', 'reentering_exits', 'reentering_days_flat', 'n_bootstrap', 'n_obs', 'n_dates_available', 'first_date', 'last_date', 'stop_fired']
regime ['vix_te

In [12]:
# the memo cites every criterion and the falsification section
memo = (ROOT / "docs" / "research" / "E10_risk_policy.md").read_text()
joined = " ".join(memo.split())
for name in ("F10.1", "F10.1b", "F10.2", "F10.2b", "F10.3", "F10.3b"):
    assert name in joined, name
assert "What would falsify this?" in joined
assert "synthetic" in joined
print("memo cites every criterion and the falsification section")

memo cites every criterion and the falsification section


In [13]:
# closing checklist: every criterion name is covered by the code
import json as _json

source = "\n".join(
    "".join(cell["source"])
    for cell in _json.loads(
        (ROOT / "notebooks" / "E10_walkthrough.ipynb").read_text()
    )["cells"]
    if cell["cell_type"] == "code"
)
assert all(
    name in source for name in ("F10.1", "F10.1b", "F10.2", "F10.2b", "F10.3", "F10.3b")
)
print("closing checklist: clean")

closing checklist: clean
